In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

config = {
    "workspace_id"  : os.getenv("WORKSPACE_ID"),
    "lakehouse_id"  : os.getenv("LAKEHOUSE_ID"),
    "storage_url"   : os.getenv("STORAGE_URL")
}

BASE_PATH = f"abfss://{config['workspace_id']}@{config['storage_url']}/{config['lakehouse_id']}"

StatementMeta(, 270a9493-fa9d-4d57-87d1-735eed1249df, 3, Finished, Available, Finished, False)

In [2]:
today_date = '2026-03-22'

StatementMeta(, 270a9493-fa9d-4d57-87d1-735eed1249df, 4, Finished, Available, Finished, False)

In [3]:
from pyspark.sql.functions import col

df = spark.read.format('delta') \
            .load(f"{BASE_PATH}/Tables/silver/tblsales_silver") \
            .filter(col('Processing_date') == today_date)

StatementMeta(, 270a9493-fa9d-4d57-87d1-735eed1249df, 5, Finished, Available, Finished, False)

In [4]:
display(df)

StatementMeta(, 270a9493-fa9d-4d57-87d1-735eed1249df, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 103d1a2e-9daa-4c46-a1db-e282aec46b1d)

In [5]:
df.printSchema()

StatementMeta(, 270a9493-fa9d-4d57-87d1-735eed1249df, 7, Finished, Available, Finished, False)

root
 |-- Row_ID: string (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Ship_Date: date (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Postal_Code: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Market: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub_Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)
 |-- Shipping_Cost: double (nullable = true)
 |-- Order_Priority: string (nullable = true)
 |-- Month: string (nu

# Gold Transformation
### Creating Dimension tables

In [6]:
from pyspark.sql.types import StringType, StructField, StructType, IntegerType, DoubleType, DateType
from delta.tables import DeltaTable

StatementMeta(, 270a9493-fa9d-4d57-87d1-735eed1249df, 8, Finished, Available, Finished, False)

##### Dim_Customer

In [7]:
# schema for dimention table
dim_customer_schema = StructType([
    StructField('Customer_ID', StringType(),True ),
    StructField('Customer_Name',StringType(),True),
    StructField('Segment',StringType(),True),
    StructField('Postal_Code',StringType(),True),
    StructField('City',StringType(),True),
    StructField('State',StringType(),True),
    StructField('Country',StringType(),True),
    StructField('Region',StringType(),True),
    StructField('Market',StringType(),True)
])

spark.sql('CREATE SCHEMA IF NOT EXISTS gold;')
DeltaTable.createIfNotExists(spark).tableName('gold.Dim_Customer').addColumns(dim_customer_schema).execute()

StatementMeta(, 270a9493-fa9d-4d57-87d1-735eed1249df, 9, Finished, Available, Finished, False)

##### Dim_Product

In [8]:
# schema for dimention table
dim_product_schema = StructType([
    StructField('Product_ID',StringType(),True),
    StructField('Category',StringType(),True),
    StructField('Sub_Category',StringType(),True),
    StructField('Product_Name',StringType(),True)
])

spark.sql('CREATE SCHEMA IF NOT EXISTS gold;')
DeltaTable.createIfNotExists(spark).tableName('gold.Dim_Product').addColumns(dim_product_schema).execute()

StatementMeta(, 270a9493-fa9d-4d57-87d1-735eed1249df, 10, Finished, Available, Finished, False)

### Creating Fact tables
##### Fact Sales table

In [9]:
# schema for fact table
fact_Sales_schema = StructType([
    StructField('Row_ID',StringType(),True),
    StructField('Order_ID',StringType(),True),
    StructField('Customer_ID',StringType(),True),
    StructField('Product_ID',StringType(),True),
    StructField('Order_Date',DateType(),True),
    StructField('Ship_Date',DateType(),True),
    StructField('Ship_Mode',StringType(),True),
    StructField('Sales',DoubleType(),True),
    StructField('Quantity',IntegerType(),True),
    StructField('Discount',DoubleType(),True),
    StructField('Profit',DoubleType(),True),
    StructField('Shipping_Cost',DoubleType(),True),
    StructField('Order_Priority',StringType(),True),
    StructField('Month',StringType(),True),
    StructField('Year',StringType(),True),
    StructField('processing_date',StringType(),True),
    StructField('Delivery_Days',StringType(),True),
    StructField('Profit_Margin',StringType(),True)
])

spark.sql('CREATE SCHEMA IF NOT EXISTS gold;')
DeltaTable.createIfNotExists(spark).tableName('gold.fact_Sales').addColumns(fact_Sales_schema).execute()

StatementMeta(, 270a9493-fa9d-4d57-87d1-735eed1249df, 11, Finished, Available, Finished, False)

### Loading data into dim_customer

In [10]:
df_selected_dim_customer = (
    df.select(
        'Customer_ID',
		'Customer_Name',
		'Segment',
		'Postal_Code',
		'City',
		'State',
		'Country',
		'Region',
		'Market'
    ).dropDuplicates(['Customer_ID'])
)

StatementMeta(, 270a9493-fa9d-4d57-87d1-735eed1249df, 12, Finished, Available, Finished, False)

In [11]:
from delta.tables import *

dim_customer_table_path = f"{BASE_PATH}/Tables/gold/dim_customer"
dim_deltacustomer = DeltaTable.forPath(spark, dim_customer_table_path)

## Perform the MERGE (UPSERT)
dim_deltacustomer.alias('target').merge(
    df_selected_dim_customer.alias('source'), 'target.Customer_ID=source.Customer_ID'
).whenMatchedUpdate(set = {
		'Customer_Name':'source.Customer_Name',
		'Segment':'source.Segment',
		'Postal_Code':'source.Postal_Code',
		'City':'source.City',
		'State':'source.State',
		'Country':'source.Country',
		'Region':'source.Region',
		'Market':'source.Market'
}).whenNotMatchedInsert(values = {
        'Customer_ID':'source.Customer_ID',
		'Customer_Name':'source.Customer_Name',
		'Segment':'source.Segment',
		'Postal_Code':'source.Postal_Code',
		'City':'source.City',
		'State':'source.State',
		'Country':'source.Country',
		'Region':'source.Region',
		'Market':'source.Market'
}).execute()

StatementMeta(, 270a9493-fa9d-4d57-87d1-735eed1249df, 13, Finished, Available, Finished, False)

In [12]:
history_df = dim_deltacustomer.history(1)  
operation_metrics = history_df.select("operationMetrics").collect()[0][0]

rows_inserted = operation_metrics.get('numTargetRowsInserted', 0)
rows_updated = operation_metrics.get('numTargetRowsUpdated', 0)
rows_deleted = operation_metrics.get('numTargetRowsDeleted', 0)

rows_affected = int(rows_inserted) + int(rows_updated) + int(rows_deleted) 

print('Total rows of table: ',dim_deltacustomer.toDF().count())
print(f"Rows inserted: {rows_inserted}")
print(f"Rows updated: {rows_updated}")
print(f"Rows deleted: {rows_deleted}")
print(f"Total rows affected: {rows_affected}")

StatementMeta(, 270a9493-fa9d-4d57-87d1-735eed1249df, 14, Finished, Available, Finished, False)

Total rows of table:  848
Rows inserted: 848
Rows updated: 0
Rows deleted: 0
Total rows affected: 848


### Loading data into dim_product

In [13]:
df_selected_dim_product = (
    df.select(
        'Product_ID',
		'Category',
		'Sub_Category',
		'Product_Name'   
    ).dropDuplicates()
)

StatementMeta(, 270a9493-fa9d-4d57-87d1-735eed1249df, 15, Finished, Available, Finished, False)

In [14]:
from delta.tables import *

dim_product_table_path = f"{BASE_PATH}/Tables/gold/dim_product"
dim_deltaproduct = DeltaTable.forPath(spark, dim_product_table_path)

dim_deltaproduct.alias('target').merge(
    df_selected_dim_product.alias('source'), 'target.Product_ID = source.Product_ID'
).whenMatchedUpdate(set = {
    'Category':'source.Category',
	'Sub_Category':'source.Sub_Category',
	'Product_Name':'source.Product_Name'
}).whenNotMatchedInsert(values = {
    'Product_ID':'source.Product_ID',
    'Category':'source.Category',
	'Sub_Category':'source.Sub_Category',
	'Product_Name':'source.Product_Name'
}).execute()

StatementMeta(, 270a9493-fa9d-4d57-87d1-735eed1249df, 16, Finished, Available, Finished, False)

In [15]:
history_df = dim_deltaproduct.history(1)  
operation_metrics = history_df.select("operationMetrics").collect()[0][0]

rows_inserted = operation_metrics.get('numTargetRowsInserted', 0)
rows_updated = operation_metrics.get('numTargetRowsUpdated', 0)
rows_deleted = operation_metrics.get('numTargetRowsDeleted', 0)

rows_affected = int(rows_inserted) + int(rows_updated) + int(rows_deleted) 

print('Total rows of table: ',dim_deltaproduct.toDF().count())
print(f"Rows inserted: {rows_inserted}")
print(f"Rows updated: {rows_updated}")
print(f"Rows deleted: {rows_deleted}")
print(f"Total rows affected: {rows_affected}")

StatementMeta(, 270a9493-fa9d-4d57-87d1-735eed1249df, 17, Finished, Available, Finished, False)

Total rows of table:  703
Rows inserted: 703
Rows updated: 0
Rows deleted: 0
Total rows affected: 703


### Loading data into Fact Sales

In [16]:
df_selected_fact_sales = (
    df.select(
        'Row_ID',
	    'Order_ID',
	    'Customer_ID',
	    'Product_ID',
	    'Order_Date',
	    'Ship_Date',
	    'Ship_Mode',
	    'Sales',
	    'Quantity',
	    'Discount',
	    'Profit',
	    'Shipping_Cost',
	    'Order_Priority',
	    'Month',
	    'Year',
	    'processing_date',
	    'Delivery_Days',
	    'Profit_Margin'
    ).dropDuplicates()
)

StatementMeta(, 270a9493-fa9d-4d57-87d1-735eed1249df, 18, Finished, Available, Finished, False)

In [17]:
from delta.tables import *

fact_sales_table_path = f"{BASE_PATH}/Tables/gold/fact_sales"
fact_deltasales = DeltaTable.forPath(spark, fact_sales_table_path)

fact_deltasales.alias('target').merge(
    df_selected_fact_sales.alias('source'),'target.Order_ID=source.Order_ID'
).whenMatchedUpdate(set = {
		    'Row_ID':'source.Row_ID',
		    'Customer_ID':'source.Customer_ID',
			'Product_ID':'source.Product_ID',
			'Order_Date':'source.Order_Date',
			'Ship_Date':'source.Ship_Date',
			'Ship_Mode':'source.Ship_Mode',
			'Sales':'source.Sales',
			'Quantity':'source.Quantity',
			'Discount':'source.Discount',
			'Profit':'source.Profit',
			'Shipping_Cost':'source.Shipping_Cost',
			'Order_Priority':'source.Order_Priority',
			'Month':'source.Month',
			'Year':'source.Year',
			'processing_date':'source.processing_date',
			'Delivery_Days':'source.Delivery_Days',
			'Profit_Margin':'source.Profit_Margin'
}).whenNotMatchedInsert(values = {
            'Row_ID':'source.Row_ID',
			'Order_ID':'source.Order_ID',
		    'Customer_ID':'source.Customer_ID',
			'Product_ID':'source.Product_ID',
			'Order_Date':'source.Order_Date',
			'Ship_Date':'source.Ship_Date',
			'Ship_Mode':'source.Ship_Mode',
			'Sales':'source.Sales',
			'Quantity':'source.Quantity',
			'Discount':'source.Discount',
			'Profit':'source.Profit',
			'Shipping_Cost':'source.Shipping_Cost',
			'Order_Priority':'source.Order_Priority',
			'Month':'source.Month',
			'Year':'source.Year',
			'processing_date':'source.processing_date',
			'Delivery_Days':'source.Delivery_Days',
			'Profit_Margin':'source.Profit_Margin'
}).execute()

StatementMeta(, 270a9493-fa9d-4d57-87d1-735eed1249df, 19, Finished, Available, Finished, False)

In [18]:
history_df = fact_deltasales.history(1)
operation_metrics = history_df.select("operationMetrics").collect()[0][0]

rows_inserted = operation_metrics.get('numTargetRowsInserted', 0)
rows_updated = operation_metrics.get('numTargetRowsUpdated', 0)
rows_deleted = operation_metrics.get('numTargetRowsDeleted', 0)

rows_affected = int(rows_inserted) + int(rows_updated) + int(rows_deleted) 

print('Total rows of table: ',fact_deltasales.toDF().count())
print(f"Rows inserted: {rows_inserted}")
print(f"Rows updated: {rows_updated}")
print(f"Rows deleted: {rows_deleted}")
print(f"Total rows affected: {rows_affected}")

StatementMeta(, 270a9493-fa9d-4d57-87d1-735eed1249df, 20, Finished, Available, Finished, False)

Total rows of table:  848
Rows inserted: 848
Rows updated: 0
Rows deleted: 0
Total rows affected: 848
